# 01 · Ingest EDGAR filings for the largest US companies

**Produces** `market_cap_raw`, `market_cap`, `market_cap_top100`, `top100_comp_sec_filings`.

Flow: SEC ticker list → market cap per ticker (Yahoo Finance, resumable batches) → top *N* companies →
their 10-K filings from the SEC submissions API.

Notes
- The universe comes from `company_tickers.json` (about 10k listed companies) instead of probing every CIK ever
  issued (about 1M requests).
- Market cap is **today's** value, so the top *N* is "largest today", not "largest when the filing was made"
  (survivorship bias, see README).
- Run cell 4 repeatedly until it reports `0 tickers left`; Yahoo throttles at about 1,000 lookups per 15 minutes.

In [ ]:
%pip install yfinance

In [ ]:
%restart_python

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
dbutils.widgets.text("sec_user_agent", "", "SEC User-Agent (name + e-mail)")
dbutils.widgets.text("batch_size", "900", "Yahoo lookups per run")
dbutils.widgets.text("top_n", "100", "Companies to keep")
dbutils.widgets.text("years", "5", "Years of filings")
dbutils.widgets.text("forms", "10-K", "Comma-separated form types")

from trading_sentiment.edgar import require_user_agent
USER_AGENT = require_user_agent(dbutils.widgets.get("sec_user_agent"))
BATCH_SIZE = int(dbutils.widgets.get("batch_size"))
TOP_N = int(dbutils.widgets.get("top_n"))
YEARS = int(dbutils.widgets.get("years"))
FORMS = tuple(f.strip() for f in dbutils.widgets.get("forms").split(","))

## 1 · Company universe from the SEC

In [ ]:
from trading_sentiment.edgar import TICKERS_URL, fetch_json, parse_company_tickers

universe = parse_company_tickers(fetch_json(TICKERS_URL, USER_AGENT))
(spark.createDataFrame(universe, "cik string, company_name string, tickers string")
      .write.format("delta").mode("overwrite").saveAsTable(table("company_universe")))
print(len(universe), "companies")

## 2 · Market cap and sector per ticker (resumable)

In [ ]:
import time
from datetime import datetime, timezone

import yfinance as yf

RAW = table("market_cap_raw")
all_tickers = sorted({r["tickers"] for r in universe if r["tickers"]})
done = {r["tickers"] for r in spark.table(RAW).select("tickers").collect()} if spark.catalog.tableExists(RAW) else set()
todo = [t for t in all_tickers if t not in done][:BATCH_SIZE]     # sorted -> every run continues where the last stopped

rows = []
for ticker in todo:
    try:
        info = yf.Ticker(ticker).info
        rows.append((ticker, info.get("marketCap"), info.get("sector"), datetime.now(timezone.utc)))
    except Exception as exc:                                        # unknown ticker, throttling, ...
        if "ratelimit" in type(exc).__name__.lower() or "too many requests" in str(exc).lower():
            # not the ticker's fault: stop and retry it next run, instead of storing a NULL that drops it from the top N
            print(f"Rate-limited by Yahoo at {ticker}; wait ~15 minutes and re-run this cell")
            break
        print(f"{ticker}: {exc}")
        rows.append((ticker, None, None, datetime.now(timezone.utc)))     # remember it so it is not retried forever
    time.sleep(0.2)

if rows:
    (spark.createDataFrame(rows, "tickers string, market_cap long, sector string, fetched_at timestamp")
          .write.format("delta").mode("append").saveAsTable(RAW))
print(f"{len(all_tickers) - len(done) - len(rows)} tickers left")

## 3 · Keep the top *N* by market cap

In [ ]:
from pyspark.sql import functions as F

market_cap = spark.table(table("market_cap_raw")).where("market_cap is not null").dropDuplicates(["tickers"])
market_cap.write.format("delta").mode("overwrite").saveAsTable(table("market_cap"))

top = market_cap.orderBy(F.desc("market_cap")).limit(TOP_N)
top.write.format("delta").mode("overwrite").saveAsTable(table("market_cap_top100"))
display(top)

## 4 · Fetch the filings of those companies

In [ ]:
from datetime import date, timedelta

from trading_sentiment.edgar import SUBMISSIONS_URL, filings_from_submissions

top_rows = spark.table(table("market_cap_top100")).join(spark.table(table("company_universe")), "tickers").collect()
since = date.today() - timedelta(days=365 * YEARS)

records = []
for row in top_rows:
    try:
        payload = fetch_json(SUBMISSIONS_URL.format(cik=row["cik"]), USER_AGENT)
    except Exception as exc:
        print(f"{row['tickers']} ({row['cik']}): {exc}")
        continue
    for filing in filings_from_submissions(payload, forms=FORMS, since=since):
        records.append({**filing, "tickers": row["tickers"], "market_cap": row["market_cap"], "sector": row["sector"]})

schema = ("cik string, company_name string, tickers string, form_type string, filing_date date, "
          "accessionNumber string, filing_url string, market_cap long, sector string")
filings = spark.createDataFrame(records, schema).orderBy(F.desc("market_cap"), F.desc("filing_date"))
filings.write.format("delta").mode("overwrite").saveAsTable(table("top100_comp_sec_filings"))
display(filings.groupBy("form_type").count())